# Round 2: gộp ứng viên và judge bằng web search

Notebook Jupyter này chạy cục bộ, gộp bảy file Round 1 và gửi mỗi câu hỏi thành một request trong OpenRouter Batch API. Judge nhận câu hỏi cùng các văn bản và Điều ứng viên, tra nguồn luật chính thức bằng native web search của OpenAI, rồi trả relevant_docs, relevant_articles, source_urls và need_review.

Batch OpenRouter không chạy công cụ web search do OpenRouter điều phối. Vì vậy notebook dùng Responses API, ghim provider OpenAI và bật native web_search. Phần model được giảm giá batch; phí web search tính theo giá thường. GPT-6 Luna hiện tính $0.01 mỗi lượt tìm kiếm. Model có thể tìm nhiều lần trong một request, vì vậy mức dự toán một search cho mỗi câu chỉ là mức sàn, không phải trần phí.

Mỗi request giới hạn tối đa hai lượt gọi công cụ tích hợp để khống chế số lượt search. Mặc định notebook chỉ lập kế hoạch 5 câu và không gửi request. Output là nhãn tạm cần người rà soát, không phải ground truth đã chốt.


In [ ]:
from pathlib import Path
from getpass import getpass
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from notebooks import round2_judge_pipeline as transport
from notebooks import round2_candidate_judge as judge

RUN_NAME = 'pilot_round2_luna_websearch_v1'
MODEL_ID = 'openai/gpt-6-luna'
LIMIT = 5
QUESTION_IDS = None
REASONING_EFFORT = 'low'
MAX_CANDIDATES = 80
MAX_REQUEST_CHARS = 50000
MAX_REQUESTS_PER_BATCH = 5
PLANNING_BUDGET_USD = 1.00
PLANNING_INPUT_USD_PER_MTOK = 0.05
PLANNING_OUTPUT_USD_PER_MTOK = 0.25
SUBMIT_BATCH = False
COLLECT_BATCHES = False

QUESTIONS_PATH = ROOT / 'data/test/questions.json'
RESULT_NAMES = (
    'vilegal_gemini-2.5-flash.json',
    'vilegal_gemini-3.1-flash-lite.json',
    'vilegal_gemini-3.5-flash-lite.json',
    'vilegal_gpt-5.6-luna.json',
    'vilegal_gpt-6-luna.json',
    'vilegal_gpt5-4-mini.json',
    'vilegal_qwen3-235b-a22b-instruct-2507.json',
)
RESULT_PATHS = [ROOT / 'data/results' / name for name in RESULT_NAMES]
RUN_DIR = ROOT / 'data/results/round2_judge' / RUN_NAME


## 1. Gộp ứng viên và xem trước

Bước offline chuẩn hóa và khử trùng relevant_docs cùng cặp law_id–article. candidate_pool.json giữ danh sách đầu vào và số phiếu Round 1 để kiểm tra thủ công; judge không thấy số phiếu, tránh neo vào đồng thuận giữa các model. Câu vượt giới hạn hoặc có dữ liệu đầu vào lỗi sẽ được đánh dấu cần rà soát.


In [ ]:
questions = transport.load_questions(QUESTIONS_PATH)
selected_ids = QUESTION_IDS if QUESTION_IDS is not None else list(questions)[:LIMIT]
plan = judge.build_plan(
    QUESTIONS_PATH, RESULT_PATHS, selected_ids,
    max_candidates=MAX_CANDIDATES, max_request_chars=MAX_REQUEST_CHARS,
    reasoning_effort=REASONING_EFFORT, model_id=MODEL_ID,
)
judge.save_plan(RUN_DIR, plan)
estimate = transport.plan_cost_ceiling(
    plan, PLANNING_INPUT_USD_PER_MTOK, PLANNING_OUTPUT_USD_PER_MTOK)
print(f'Đã chọn {len(selected_ids)} câu; {len(plan["requests"])} câu có request.')
print('Dự toán token và reserve cho tối đa hai lượt gọi search:', estimate)
print('max_tool_calls chặn ở hai lượt công cụ mỗi câu; dự toán token vẫn là proxy, nên kiểm tra usage.cost sau batch.')
print('Danh sách ứng viên:', RUN_DIR / 'candidate_pool.json')
print('Kế hoạch:', RUN_DIR / 'plan.json')
for qid in selected_ids[:10]:
    case = plan['cases'][str(qid)]
    print(f'ID {qid}: {len(case["candidate_docs"])} văn bản, '
          f'{len(case["candidate_articles"])} Điều, {len(case["review_reasons"])} lý do rà')
if plan['requests']:
    first = plan['requests'][0]
    print('Request đầu:', first['custom_id'],
          'bytes:', len(__import__('json').dumps(first['body'], ensure_ascii=False).encode('utf-8')))


## 2. Nộp batch OpenRouter

Đặt SUBMIT_BATCH = True rồi chạy cell kế tiếp. Notebook hỏi API key bằng getpass nên không ghi key vào notebook hay output. Batch gửi tối đa MAX_REQUESTS_PER_BATCH câu chưa nộp và ghim provider OpenAI để native web search hoạt động. Mỗi lần chỉ nên gửi pilot nhỏ, đợi kết quả rồi xem chi phí thực tế trước khi tăng batch. Nếu kết nối mất sau khi gửi mà chưa nhận batch ID, state được giữ ở submission_uncertain để tránh nộp trùng.


In [ ]:
if SUBMIT_BATCH:
    api_key = getpass('OpenRouter API key: ')
    try:
        batch = transport.submit_next_batch(
            RUN_DIR, api_key, MAX_REQUESTS_PER_BATCH,
            budget_usd=PLANNING_BUDGET_USD,
            input_usd_per_mtok=PLANNING_INPUT_USD_PER_MTOK,
            output_usd_per_mtok=PLANNING_OUTPUT_USD_PER_MTOK,
        )
        print('Đã nộp:', batch if batch else 'Không còn request mới')
    finally:
        del api_key
else:
    print('Chế độ xem trước; chưa gọi API.')


## 3. Thu kết quả và xuất nhãn tạm

Sau khi batch hoàn tất, đặt COLLECT_BATCHES = True. Cell kế tiếp cập nhật trạng thái, ghép response theo custom_id, lưu usage và xuất provisional_labels.json. Có thể chạy lại cell để nhận kết quả batch đang xử lý. Những URL judge trả về được đối chiếu với URL citation trong response và chỉ giữ nguồn HTTPS thuộc danh sách domain chính thức. Kết quả thiếu nguồn, không hợp lệ hoặc không chắc chắn được đưa vào need_review.


In [ ]:
if COLLECT_BATCHES:
    api_key = getpass('OpenRouter API key: ')
    try:
        state = transport.collect_batches(RUN_DIR, api_key)
        print('Batch:', [(item['batch_id'], item['status']) for item in state['batches']])
        print('Chi phí/token OpenRouter báo hoặc còn dự phòng:', transport.cost_report(state))
    finally:
        del api_key
else:
    print('Chưa truy vấn trạng thái batch.')
labels = judge.export_labels(RUN_DIR)
print('Nhãn tạm:', len(labels), 'câu; cần rà:', sum(row['need_review'] for row in labels))
print('Danh sách ứng viên:', RUN_DIR / 'candidate_pool.json')
print('Nhãn tạm:', RUN_DIR / 'provisional_labels.json')
